# Data Cleaning — SAP News Intelligence

**Inputs** (all in `notebook/data/`):
| File | Source | Rows | Notes |
|---|---|---|---|
| `sap_news.json` | SAP News RSS + scraper | 150 | Full article content available |
| `gnews_articles.json` | GNews API | 43 | Content truncated by API; use description |
| `hackernews_posts.json` | Hacker News | 220 | 107/220 rows have article content |

**Output:** `notebook/data/clean_data.json`


## 1. Imports & configuration

In [1]:
import re
import unicodedata
from pathlib import Path

import pandas as pd
from rapidfuzz import fuzz

# ── Resolve notebook/data/ regardless of where Jupyter sets CWD ─────────────
_cwd = Path.cwd()
candidates = [
    _cwd / "data",
    _cwd / "notebook" / "data",
    _cwd.parent / "data",
]
DATA_DIR = next((p for p in candidates if p.exists()), candidates[0])

SAP_PATH   = DATA_DIR / "sap_news.json"
GNEWS_PATH = DATA_DIR / "gnews_articles.json"
HN_PATH    = DATA_DIR / "hackernews_posts.json"
OUT_PATH   = DATA_DIR / "clean_data.json"

MIN_TEXT_LEN    = 100
FUZZY_THRESHOLD = 85

print(f"Working dir : {_cwd}")
print(f"Data dir    : {DATA_DIR}  (exists: {DATA_DIR.exists()})")
for p in [SAP_PATH, GNEWS_PATH, HN_PATH]:
    print(f"  {p.name}: {p.exists()}")


Working dir : /workspaces/AI-CEO-Strategic-Intelligence-Agent/notebook
Data dir    : /workspaces/AI-CEO-Strategic-Intelligence-Agent/notebook/data  (exists: True)
  sap_news.json: True
  gnews_articles.json: True
  hackernews_posts.json: True


## 2. Load raw sources

In [2]:
sap_df   = pd.read_json(SAP_PATH)
gnews_df = pd.read_json(GNEWS_PATH)
hn_df    = pd.read_json(HN_PATH)

print(f"SAP News     : {sap_df.shape}   columns: {sap_df.columns.tolist()}")
print(f"GNews        : {gnews_df.shape}  columns: {gnews_df.columns.tolist()}")
print(f"Hacker News  : {hn_df.shape}  columns: {hn_df.columns.tolist()}")


SAP News     : (150, 6)   columns: ['title', 'url', 'published', 'source', 'description', 'content']
GNews        : (45, 6)  columns: ['title', 'description', 'content', 'source', 'published', 'url']
Hacker News  : (220, 6)  columns: ['title', 'description', 'content', 'published', 'source', 'url']


## 3. Build `text` field per source

- **SAP News**: full content available — use `title + description + content`
- **GNews**: content is truncated by the API to ~266 chars — strip the marker, use what's there
- **Hacker News**: description is always empty; 107/220 rows have content — use whatever exists


In [3]:
COLS = ["title", "description", "content", "source", "published", "url"]

sap_clean = sap_df[COLS].copy()
sap_clean["text"] = (
    sap_clean["title"].fillna("") + " "
    + sap_clean["description"].fillna("") + " "
    + sap_clean["content"].fillna("")
)

gnews_clean = gnews_df[COLS].copy()
gnews_clean["content_stripped"] = gnews_clean["content"].str.replace(
    r"\[\+\d+\s*chars\]", "", regex=True
).str.strip()
gnews_clean["text"] = (
    gnews_clean["title"].fillna("") + " "
    + gnews_clean["description"].fillna("") + " "
    + gnews_clean["content_stripped"].fillna("")
)
gnews_clean = gnews_clean.drop(columns=["content_stripped"])

hn_clean = hn_df[COLS].copy()
hn_clean["text"] = (
    hn_clean["title"].fillna("") + " "
    + hn_clean["content"].fillna("")
)

print(f"SAP text length   — mean: {sap_clean['text'].str.len().mean():.0f}, min: {sap_clean['text'].str.len().min()}")
print(f"GNews text length — mean: {gnews_clean['text'].str.len().mean():.0f}, min: {gnews_clean['text'].str.len().min()}")
print(f"HN text length    — mean: {hn_clean['text'].str.len().mean():.0f}, min: {hn_clean['text'].str.len().min()}")


SAP text length   — mean: 6596, min: 60
GNews text length — mean: 616, min: 394
HN text length    — mean: 392, min: 12


## 4. Merge all sources

In [4]:
all_docs = pd.concat([sap_clean, gnews_clean, hn_clean], ignore_index=True)

print(f"Combined shape: {all_docs.shape}")
print(f"Sources:\n{all_docs['source'].value_counts().to_string()}")


Combined shape: (415, 7)
Sources:
source
Hacker News                    220
SAP News                       150
The Manila Times                 6
The Tribune                      6
The Economic Times               5
TechBullion                      4
MarketScreener                   3
The Hindu Business Line          3
PR Newswire UK                   3
scanx.trade                      2
GlobeNewswire                    1
TNW                              1
Devdiscourse                     1
The Hitavada                     1
The Hindu                        1
CNBC TV18                        1
The Financial Express            1
CP24 Toronto                     1
The Star                         1
India Today                      1
The Irish Times                  1
Australian Financial Review      1
Livemint                         1


## 5. Clean text

- Normalize unicode (NFKC)
- Remove invisible characters
- Strip stray HTML tags
- Remove URLs and email addresses
- Remove GNews truncation markers (`[+N chars]`)
- Remove repeated separators (`***`, `---`, `===`, `>>>`)
- Collapse excess whitespace


In [5]:
def clean_text(text: str) -> str:
    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"[\u200b\u200c\u200d\u200e\u200f\u2060\ufeff\u2028\u2029]", " ", text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+", "", text)
    text = re.sub(r"[\w.+-]+@[\w.-]+\.\w+", "", text)
    text = re.sub(r"\[\+\d+\s*chars\]", "", text)
    text = re.sub(r"[*]{3,}|[>]{3,}|[-]{3,}|[=]{3,}|[#]{3,}", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


all_docs["clean_text"]   = all_docs["text"].apply(clean_text)
all_docs["clean_length"] = all_docs["clean_text"].str.len()

print(all_docs["clean_length"].describe())


count      415.000000
mean      2633.190361
std       3838.216931
min         11.000000
25%         75.000000
50%        600.000000
75%       5116.000000
max      45460.000000
Name: clean_length, dtype: float64


## 6. Parse & sort dates

In [6]:
all_docs["published"] = pd.to_datetime(all_docs["published"], errors="coerce", utc=True, format="mixed")

n_null = all_docs["published"].isna().sum()
print(f"Unparseable dates : {n_null} (kept as NaT)")
print(f"Date range        : {all_docs['published'].min()}  →  {all_docs['published'].max()}")

all_docs = all_docs.sort_values("published", na_position="last").reset_index(drop=True)


Unparseable dates : 265 (kept as NaT)
Date range        : 2026-02-02 13:15:00+00:00  →  2026-06-19 10:15:00+00:00


## 7. Filter off-topic / irrelevant articles

Two filters:

1. **Known junk titles** — a manually curated list of completely unrelated
   articles (skincare ingredients, gaming laptop roundups, etc.) that slipped
   through the collection queries.
2. **SAP relevance filter** — the original collection queries included
   broad terms like `"enterprise software"` and `"business AI"` (GNews) and
   a bare `"enterprise software"` query (Hacker News full-text search). These
   pulled in articles that never mention SAP at all — e.g. *"SpaceX to buy AI
   coding startup Cursor"*, *"I Bought a 'Junk' PSP From Japan"*, generic
   "Ask HN" threads about enterprise software with no SAP connection.
   We drop any row where "SAP" (whole word, case-insensitive) doesn't appear
   anywhere in the combined text.


In [7]:
IRRELEVANT_TITLES = [
    "What is dragon's blood resin? The forgotten 2,000-year-old skincare ingredient used by ancient Roman and Arab women",
    "Tech updates (June 11, 2026): New ASUS gaming laptops, Canva Offline, Samsung AI deals",
    "Business wisdom of the day: 'The bamboo that bends is stronger than the oak that resists'",
    "India emerges as world's top marketing",
]

before = len(all_docs)
all_docs = all_docs[~all_docs["title"].isin(IRRELEVANT_TITLES)].reset_index(drop=True)
print(f"Removed {before - len(all_docs)} known off-topic articles")

# SAP relevance filter — must mention "SAP" as a whole word somewhere in the text
before = len(all_docs)
mentions_sap = all_docs["text"].str.contains(r"\bSAP\b", case=False, regex=True, na=False)
dropped = all_docs[~mentions_sap][["source", "title"]]
all_docs = all_docs[mentions_sap].reset_index(drop=True)

print(f"Removed {before - len(all_docs)} articles with no SAP mention  ({before} → {len(all_docs)})")
print("\nDropped articles (by source):")
print(dropped["source"].value_counts().to_string())


Removed 0 known off-topic articles
Removed 102 articles with no SAP mention  (415 → 313)

Dropped articles (by source):
source
Hacker News                    80
The Economic Times              4
The Manila Times                3
The Tribune                     2
MarketScreener                  2
TechBullion                     2
PR Newswire UK                  2
SAP News                        1
CP24 Toronto                    1
The Star                        1
India Today                     1
The Irish Times                 1
Australian Financial Review     1
Livemint                        1


## 8. Drop short articles

In [8]:
before = len(all_docs)
all_docs = all_docs[all_docs["clean_length"] >= MIN_TEXT_LEN].reset_index(drop=True)
print(f"Dropped {before - len(all_docs)} articles shorter than {MIN_TEXT_LEN} chars  ({before} → {len(all_docs)})")


Dropped 48 articles shorter than 100 chars  (313 → 265)


## 9. Deduplicate

1. **Exact** — same title
2. **Fuzzy** — titles with similarity ≥ 85 (catches rephrased duplicates)


In [9]:
before = len(all_docs)
all_docs = all_docs.drop_duplicates(subset=["title"]).reset_index(drop=True)
print(f"After exact dedup : {len(all_docs)}  (removed {before - len(all_docs)})")

titles = all_docs["title"].fillna("").tolist()
to_drop = set()
for i in range(len(titles)):
    if i in to_drop:
        continue
    for j in range(i + 1, len(titles)):
        if fuzz.ratio(titles[i], titles[j]) >= FUZZY_THRESHOLD:
            to_drop.add(j)

all_docs = all_docs.drop(index=list(to_drop)).reset_index(drop=True)
print(f"After fuzzy dedup : {len(all_docs)}  (removed {len(to_drop)} near-duplicates)")


After exact dedup : 255  (removed 10)
After fuzzy dedup : 250  (removed 5 near-duplicates)


## 10. Add `data_source_type` column

In [10]:
all_docs["data_source_type"] = all_docs["source"].apply(
    lambda s: "owned" if s == "SAP News" else "third_party"
)
print(all_docs["data_source_type"].value_counts().to_string())


data_source_type
owned          148
third_party    102


## 11. Final summary

In [11]:
print("=" * 50)
print("  DATA CLEANING SUMMARY")
print("=" * 50)
print(f"  Total articles      : {len(all_docs)}")
print(f"  Columns             : {all_docs.columns.tolist()}")
print(f"  SAP News (owned)    : {(all_docs['data_source_type']=='owned').sum()}")
print(f"  Third-party         : {(all_docs['data_source_type']=='third_party').sum()}")
print(f"  Missing dates (NaT) : {all_docs['published'].isna().sum()}")
print(f"  Min clean_length    : {all_docs['clean_length'].min()}")
print(f"  Max clean_length    : {all_docs['clean_length'].max()}")
print(f"  Mean clean_length   : {all_docs['clean_length'].mean():.0f}")
print("=" * 50)
all_docs[["title","source","published","clean_length","data_source_type"]].head(5)


  DATA CLEANING SUMMARY
  Total articles      : 250
  Columns             : ['title', 'description', 'content', 'source', 'published', 'url', 'text', 'clean_text', 'clean_length', 'data_source_type']
  SAP News (owned)    : 148
  Third-party         : 102
  Missing dates (NaT) : 102
  Min clean_length    : 137
  Max clean_length    : 45460
  Mean clean_length   : 4195


,title,source,published,clean_length,data_source_type
0,From Intent to Impact: How Supply Chain Leader...,SAP News,2026-02-02 13:15:00+00:00,6372,owned
1,"At Davos 2026, Sustainability Was Everywhere, ...",SAP News,2026-02-04 12:15:00+00:00,8832,owned
2,SAP Introduces New Capabilities to Advance Pay...,SAP News,2026-02-04 13:15:00+00:00,5803,owned
3,SAP Is a Leader in the 2025 Gartner® Magic Qua...,SAP News,2026-02-05 12:15:00+00:00,4028,owned
4,"AI, Sustainability, and the New Blueprint for ...",SAP News,2026-02-05 13:15:00+00:00,7653,owned


## 12. Save `clean_data.json`

In [12]:
all_docs.to_json(OUT_PATH, orient="records", indent=4, date_format="iso")
print(f"Saved → {OUT_PATH}")
print(f"Shape  : {all_docs.shape}")


Saved → /workspaces/AI-CEO-Strategic-Intelligence-Agent/notebook/data/clean_data.json
Shape  : (250, 10)
